In [ ]:
## problem 
##South African provinces generate large amounts of waste, but waste management remains reactive and inefficient. 
#There is no clear system to predict future waste volumes, limiting effective planning and resource allocation. 
# This project uses linear regression to predict total waste generated per province, helping municipalities anticipate waste trends and improve sustainability efforts.



## mission
##To support better waste management through data-driven insights. 
#This project aims to build a simple predictive model that estimates waste generation across South Africa, 
#helping optimize waste reduction, reuse, recycling, and recovery activities for a more sustainable future.


# STEP 1: Install (Colab typically has these pre-installed)
!pip install scikit-learn pandas matplotlib seaborn joblib

# STEP 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib


# Replace the path with your Kaggle file or uploaded file
df = pd.read_csv('/South_African_Waste_Data.csv')

# Drop empty unnamed column
df.drop(['Unnamed: 6'], axis=1, inplace=True)

# Drop empty rows
df.dropna(how='all', inplace=True)

# Clean numeric columns: remove commas, convert to float
for col in ['General Waste (t)', 'Hazardous Waste (t)', 'Total Tonnage (t)']:
    df[col] = df[col].astype(str).str.replace(",", "").astype(float)

# Drop Year (same year in all rows)
df.drop(['Year'], axis=1, inplace=True)

# Visual check
df.head()
df.info()




In [ ]:

# STEP 4: Data Visualization

plt.figure(figsize=(12,6))
sns.barplot(data=df, x='Province', y='Total Tonnage (t)', estimator=sum)
plt.xticks(rotation=45)
plt.title('Total Waste Tonnage per Province')
plt.show()


In [ ]:
# STEP 5: Prepare Data for Modeling

# One-hot encode Province
df_encoded = pd.get_dummies(df, columns=['Province'], drop_first=True)

# Features & Target
X = df_encoded.drop('Total Tonnage (t)', axis=1)
y = df_encoded['Total Tonnage (t)']

# Standardize Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)


In [ ]:
# STEP 6: Linear Regression (Using SGDRegressor)

lr = SGDRegressor(max_iter=1000, learning_rate='invscaling', eta0=0.01, random_state=42)
losses = []

for epoch in range(1, 101):
    lr.partial_fit(X_train, y_train)
    y_train_pred = lr.predict(X_train)
    mse = mean_squared_error(y_train, y_train_pred)
    losses.append(mse)

# Plot Loss Curve
plt.figure(figsize=(10,5))
plt.plot(losses)
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.title('Loss Curve (Training Data)')
plt.show()

# Evaluate Linear Regression
y_test_pred_lr = lr.predict(X_test)
print("Linear Regression Test RMSE:", mean_squared_error(y_test, y_test_pred_lr))
print("Linear Regression Test R²:", r2_score(y_test, y_test_pred_lr))


In [ ]:
# STEP 7: Model Comparison

# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_test_pred_dt = dt.predict(X_test)

# Random Forest
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_test_pred_rf = rf.predict(X_test)

# Print Results
print("\nDecision Tree RMSE:", mean_squared_error(y_test, y_test_pred_dt))
print("Decision Tree R²:", r2_score(y_test, y_test_pred_dt))

print("\nRandom Forest RMSE:", mean_squared_error(y_test, y_test_pred_rf))
print("Random Forest R²:", r2_score(y_test, y_test_pred_rf))


In [ ]:
# STEP 8: Save Best Performing Model

# Assuming Random Forest performed best
joblib.dump(rf, 'best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Download models (optional)
from google.colab import files
files.download('best_model.pkl')
files.download('scaler.pkl')


In [ ]:
# STEP 9: Simple Prediction Function (for Task 2)

def predict_waste(input_data):
    model = joblib.load('best_model.pkl')
    scaler = joblib.load('scaler.pkl')
    input_scaled = scaler.transform([input_data])
    prediction = model.predict(input_scaled)
    return prediction[0]
